# 🤖 Model Training - Crypto Pulse

This notebook covers:
1. **Sentiment Analysis** using FinBERT.
2. **Price Forecasting** using LSTM.

## 1. Setup and Environment

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import glob
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from transformers import pipeline
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

print(f"PyTorch version: {torch.__version__}")

## 2. Sentiment Analysis (Task 2.1)

We use **FinBERT** for market sentiment analysis.

In [ ]:
sentiment_pipe = pipeline("text-classification", model="ProsusAI/finbert")

def get_sentiment_score(text):
    if not text or pd.isna(text): return 'neutral', 0.0
    result = sentiment_pipe(text[:512])[0]
    return result['label'], result['score']

## 3. Price Forecasting (Task 2.2)

We will build a **Long Short-Term Memory (LSTM)** network to predict future prices.

In [ ]:
def prepare_lstm_data(df, lookback=60):
    data = df['close'].values.reshape(-1, 1)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)
    
    X, y = [], []
    for i in range(lookback, len(scaled_data)):
        X.append(scaled_data[i-lookback:i, 0])
        y.append(scaled_data[i, 0])
        
    return np.array(X), np.array(y), scaler

class CryptoLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2):
        super(CryptoLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

print("LSTM Model Architecture Ready.")